# pclass와 age 그리고 gender를 기준으로 전처리

- Surived:0=사망, 1=생존
- Pclass: 1=1등석, 2=2등석, 3=3등석
- gender:male=남성, female=여성
- Age: 나이
- SibSp: 타이타닉 호에 동승한 자매/배우자의 수
- Parch: 타이타닉 호에 동승한 부모/자식의 수
- Ticket: 티켓 번호 X
- Fare: 승객 요금 X
- Embarked: 탑승지; C=셰르부르, Q=퀴즈타운, S=사우샘프턴 => 상관관계 X

In [233]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

# train data
df_train = pd.read_csv("./train.csv", encoding="utf-8")

# test data
df_test = pd.read_csv("./test.csv", encoding="utf-8")

df_train.info(), df_train.shape

<class 'pandas.DataFrame'>
RangeIndex: 916 entries, 0 to 915
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   passengerid  916 non-null    int64  
 1   survived     916 non-null    int64  
 2   pclass       916 non-null    int64  
 3   name         916 non-null    str    
 4   gender       916 non-null    str    
 5   age          736 non-null    float64
 6   sibsp        916 non-null    int64  
 7   parch        916 non-null    int64  
 8   ticket       916 non-null    str    
 9   fare         916 non-null    float64
 10  cabin        198 non-null    str    
 11  embarked     915 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 86.0 KB


(None, (916, 12))

In [234]:
df_test.info(), df_test.shape   

<class 'pandas.DataFrame'>
RangeIndex: 393 entries, 0 to 392
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   passengerid  393 non-null    int64  
 1   pclass       393 non-null    int64  
 2   name         393 non-null    str    
 3   gender       393 non-null    str    
 4   age          310 non-null    float64
 5   sibsp        393 non-null    int64  
 6   parch        393 non-null    int64  
 7   ticket       393 non-null    str    
 8   fare         392 non-null    float64
 9   cabin        97 non-null     str    
 10  embarked     392 non-null    str    
dtypes: float64(2), int64(4), str(5)
memory usage: 33.9 KB


(None, (393, 11))

In [235]:
df_test_copy = df_test.copy()
df_train_copy = df_train.copy()

In [236]:
df_test_copy.shape, df_test_copy.shape

((393, 11), (393, 11))

In [237]:
df_train_copy.drop(columns=["cabin","embarked","ticket","fare","name", 'sibsp', 'parch'],inplace=True)
df_test_copy.drop(columns=["cabin","embarked","ticket","fare","name", 'sibsp', 'parch'],inplace=True)

In [238]:
df_test_copy.columns, df_test_copy.shape

(Index(['passengerid', 'pclass', 'gender', 'age'], dtype='str'), (393, 4))

In [222]:
bins = [0,10,20,30,40,50,60,70,80]
# age들을 묶을 카테고리 설정
labels = ["0-10","10-20","20-30","30-40","40-50","50-60","60-70","70-80"]

# age_group 이라는 이름을 추가해서 나이대별을 10씩 묶음 EX) 13->10-20 / 23->20-30 
df_test_copy["age_group"] = pd.cut(df_test_copy["age"], bins=bins, labels=labels)
df_train_copy["age_group"] = pd.cut(df_train_copy["age"], bins=bins, labels=labels)

In [239]:
df_train_copy.head()

,passengerid,survived,pclass,gender,age
0,0,0,2,male,NaN
1,1,0,3,female,NaN
2,2,1,1,female,52.0
3,3,1,3,male,27.0
4,4,0,2,male,44.0


In [240]:
df_train_copy["gender"] = df_train_copy["gender"].map({
    "male":0,
    "female":1
})
df_test_copy["gender"] = df_test_copy["gender"].map({
    "male":0,
    "female":1
})

In [231]:
# df_group이라는 데이터 프레임으로 복사 후 저장
# df_train_copy = df_train_copy.drop(columns = ["age_group"])
# df_test_copy = df_test_copy.drop(columns = ["age_group"])
# = df_train_copy[["gender","age_group","survived","pclass"]].copy()

In [241]:
df_train_copy.head()

,passengerid,survived,pclass,gender,age
0,0,0,2,0,NaN
1,1,0,3,1,NaN
2,2,1,1,1,52.0
3,3,1,3,0,27.0
4,4,0,2,0,44.0


In [242]:
df_train_copy["age"] = df_train_copy["age"].fillna(
    df_train_copy.groupby(["pclass", "gender"])["age"].transform("median")
)
df_test_copy["age"] = df_test_copy["age"].fillna(
    df_test_copy.groupby(["pclass", "gender"])["age"].transform("median")
)

In [243]:
df_train_copy.head()

,passengerid,survived,pclass,gender,age
0,0,0,2,0,30.0
1,1,0,3,1,22.0
2,2,1,1,1,52.0
3,3,1,3,0,27.0
4,4,0,2,0,44.0


In [244]:
df_train_copy["age_range"] = pd.cut(
    df_train_copy["age"],
    bins=[0,10,20,30,40,50,60,80],
    labels=[0,1,2,3,4,5,6]
)
df_test_copy["age_range"] = pd.cut(
    df_test_copy["age"],
    bins=[0,10,20,30,40,50,60,80],
    labels=[0,1,2,3,4,5,6]
)

In [246]:
df_train_copy.head()

,passengerid,survived,pclass,gender,age,age_range
0,0,0,2,0,30.0,2
1,1,0,3,1,22.0,2
2,2,1,1,1,52.0,5
3,3,1,3,0,27.0,2
4,4,0,2,0,44.0,4


In [245]:
df_train_copy.isnull().sum().sum()

np.int64(0)

In [247]:
df_test_copy.isnull().sum().sum()

np.int64(0)

In [248]:
df_train_copy.head()

,passengerid,survived,pclass,gender,age,age_range
0,0,0,2,0,30.0,2
1,1,0,3,1,22.0,2
2,2,1,1,1,52.0,5
3,3,1,3,0,27.0,2
4,4,0,2,0,44.0,4


# 여기까지